# 01 Data Cleaning

This notebook sets up the initial cleaned datasets for the project using the workbook `annual-energy-consumption-data-2024_cleaning.xlsx`.

For now, the notebook does four things:
1. Loads the Excel workbook with pandas.
2. Creates a `properties` dataframe from the `Properties` sheet using only the required columns.
3. Creates separate gas and electric meter-entry dataframes using only the required columns.
4. Merges the property metadata into the gas and electric dataframes using `Property Name` and `Portfolio Manager ID`.

## Import libraries and define the file path

This cell imports pandas, sets the workbook path, and confirms the available sheet names before we start selecting columns.

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

file_path = Path("../data/clean/annual-energy-consumption-data-2024_cleaning.xlsx")

excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

['Properties', 'Meter Entries - Gas', 'Meter Entries - Electeric']

## Create the `properties` dataframe

From the first worksheet, `Properties`, we keep only these columns:
- `A`: `Property Name`
- `B`: `Portfolio Manager ID`
- `H`: `Property Type - Self-Selected`
- `I`: `Gross Floor Area`

These are the property-level fields needed later for analysis and modeling.

In [3]:
properties = pd.read_excel(
    file_path,
    sheet_name="Properties",
    usecols="A,B,H,I",
)

properties.head()

,Property Name,Portfolio Manager ID,Property Type - Self-Selected,Gross Floor Area
0,F.J. Horgan Water Treatment Plant,35000838,Drinking Water Treatment & Distribution,325447
1,Island Water Treatment Plant,35000839,Drinking Water Treatment & Distribution,64196
2,W.H. Johnston Pumping Station,35000836,Drinking Water Treatment & Distribution,1744
3,West Toronto Pumping Station,35000837,Drinking Water Treatment & Distribution,7739
4,St. Albans Pumping Station,35000834,Drinking Water Treatment & Distribution,3240


## Create the gas meter dataframe

From the second worksheet, `Meter Entries - Gas`, we keep the requested columns:
- `A`: `Property Name`
- `B`: `Portfolio Manager ID`
- `E`: `Meter Type`
- `G`: `Start Date`
- `H`: `End Date`
- `I`: `Usage/Quantity`
- `J`: `Usage Units`
- `K`: `Cost ($)`

This dataframe is named `gas_entries`.

In [4]:
gas_entries = pd.read_excel(
    file_path,
    sheet_name="Meter Entries - Gas",
    usecols="A,B,E,G,H,I,J,K",
)

gas_entries.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($)
0,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-01-01,2024-02-01,23839.31,cm (cubic meters),10317.96
1,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-02-01,2024-03-01,18735.80,cm (cubic meters),8181.12
2,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-03-01,2024-04-01,14082.81,cm (cubic meters),7422.93
3,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-04-01,2024-05-01,11537.50,cm (cubic meters),5674.99
4,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-05-01,2024-06-01,7784.48,cm (cubic meters),3675.73


## Create the electric meter dataframe

From the third worksheet, `Meter Entries - Electeric`, we keep the requested columns:
- `A`: `Property Name`
- `B`: `Portfolio Manager ID`
- `E`: `Meter Type`
- `G`: `Start Date`
- `H`: `End Date`
- `I`: `Usage/Quantity`
- `J`: `Usage Units`
- `K`: `Cost ($)`

This dataframe is named `electric_entries`.

In [5]:
electric_entries = pd.read_excel(
    file_path,
    sheet_name="Meter Entries - Electeric",
    usecols="A,B,E,G,H,I,J,K",
)

electric_entries.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($)
0,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-01-01,2024-02-01,3294191.84,kWh (thousand Watt-hours),335409.93
1,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-02-01,2024-03-01,3188060.05,kWh (thousand Watt-hours),300680.85
2,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-03-01,2024-04-01,2971166.22,kWh (thousand Watt-hours),273185.01
3,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-04-01,2024-05-01,2861591.94,kWh (thousand Watt-hours),255243.37
4,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-05-01,2024-06-01,3142472.63,kWh (thousand Watt-hours),285188.75


## Merge property information into the gas and electric dataframes

Now we attach the property-level information to both meter-entry datasets.

The merge uses both:
- `Property Name`
- `Portfolio Manager ID`

Using both columns makes the join explicit and keeps the property information aligned with the matching meter records.

In [6]:
merge_keys = ["Property Name", "Portfolio Manager ID"]

gas_with_properties = gas_entries.merge(
    properties,
    on=merge_keys,
    how="left",
)

gas_with_properties.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($),Property Type - Self-Selected,Gross Floor Area
0,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-01-01,2024-02-01,23839.31,cm (cubic meters),10317.96,Drinking Water Treatment & Distribution,325447
1,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-02-01,2024-03-01,18735.80,cm (cubic meters),8181.12,Drinking Water Treatment & Distribution,325447
2,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-03-01,2024-04-01,14082.81,cm (cubic meters),7422.93,Drinking Water Treatment & Distribution,325447
3,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-04-01,2024-05-01,11537.50,cm (cubic meters),5674.99,Drinking Water Treatment & Distribution,325447
4,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-05-01,2024-06-01,7784.48,cm (cubic meters),3675.73,Drinking Water Treatment & Distribution,325447


In [7]:
electric_with_properties = electric_entries.merge(
    properties,
    on=merge_keys,
    how="left",
)

electric_with_properties.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($),Property Type - Self-Selected,Gross Floor Area
0,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-01-01,2024-02-01,3294191.84,kWh (thousand Watt-hours),335409.93,Drinking Water Treatment & Distribution,325447
1,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-02-01,2024-03-01,3188060.05,kWh (thousand Watt-hours),300680.85,Drinking Water Treatment & Distribution,325447
2,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-03-01,2024-04-01,2971166.22,kWh (thousand Watt-hours),273185.01,Drinking Water Treatment & Distribution,325447
3,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-04-01,2024-05-01,2861591.94,kWh (thousand Watt-hours),255243.37,Drinking Water Treatment & Distribution,325447
4,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-05-01,2024-06-01,3142472.63,kWh (thousand Watt-hours),285188.75,Drinking Water Treatment & Distribution,325447


## Remove duplicate rows from the merged dataframes

After merging, some rows in gas_with_properties and electric_with_properties are duplicated.

This section removes fully duplicated rows from both merged dataframes and then prints the updated shapes so we can confirm the result.

In [9]:
gas_duplicates_before = gas_with_properties.duplicated().sum()
electric_duplicates_before = electric_with_properties.duplicated().sum()

gas_with_properties = gas_with_properties.drop_duplicates().reset_index(drop=True)
electric_with_properties = electric_with_properties.drop_duplicates().reset_index(drop=True)

print("Duplicate rows removed from gas_with_properties:", gas_duplicates_before)
print("Duplicate rows removed from electric_with_properties:", electric_duplicates_before)

print("\nNew shapes after removing duplicates:")
print("gas_with_properties:", gas_with_properties.shape)
print("electric_with_properties:", electric_with_properties.shape)

Duplicate rows removed from gas_with_properties: 3512
Duplicate rows removed from electric_with_properties: 4578

New shapes after removing duplicates:
gas_with_properties: (8522, 10)
electric_with_properties: (19832, 10)


In [10]:
dataframes_with_building_type = {
    'properties': properties,
    'gas_with_properties': gas_with_properties,
    'electric_with_properties': electric_with_properties,
}

for df_name, df in dataframes_with_building_type.items():
    print(f'\n{df_name} building type counts:')
    display(
        df['Property Type - Self-Selected']
        .value_counts(dropna=False)
        .rename_axis('Property Type - Self-Selected')
        .reset_index(name='Row Count')
    )


properties building type counts:


,Property Type - Self-Selected,Row Count
0,Other - Public Services,880
1,Transportation Terminal/Station,169
2,Parking,138
3,Fire Station,92
4,Library,90
5,Community Center and Social Meeting Hall,77
6,Office,53
7,Other - Recreation,51
8,Wastewater Treatment Plant,48
9,Police Station,37



gas_with_properties building type counts:


,Property Type - Self-Selected,Row Count
0,Other - Public Services,3084
1,Fire Station,1080
2,Library,993
3,Community Center and Social Meeting Hall,803
4,Other - Recreation,534
5,Office,516
6,Police Station,379
7,Indoor Arena,318
8,Transportation Terminal/Station,214
9,Other - Entertainment/Public Assembly,205



electric_with_properties building type counts:


,Property Type - Self-Selected,Row Count
0,Other - Public Services,9859
1,Transportation Terminal/Station,1962
2,Parking,1648
3,Fire Station,1092
4,Library,1052
5,Community Center and Social Meeting Hall,924
6,Other - Recreation,600
7,Office,598
8,Wastewater Treatment Plant,564
9,Police Station,421


In [11]:
building_type_mapping = {
    'Fire Station': 'Public Safety',
    'Police Station': 'Public Safety',
    'Indoor Arena': 'Recreation & Community',
    'Community Center and Social Meeting Hall': 'Recreation & Community',
    'Other - Recreation': 'Recreation & Community',
    'Other - Entertainment/Public Assembly': 'Recreation & Community',
    'Office': 'Administrative / Office',
    'Mixed Use Property': 'Administrative / Office',
    'Library': 'Education & Social Services',
    'Pre-school/Daycare': 'Education & Social Services',
    'Residential Care Facility': 'Education & Social Services',
    'Parking': 'Transportation',
    'Transportation Terminal/Station': 'Transportation',
    'Drinking Water Treatment & Distribution': 'Utilities / Infrastructure',
    'Wastewater Treatment Plant': 'Utilities / Infrastructure',
    'Other': 'Other-Public Services',
    'Other - Public Services': 'Other-Public Services',
}

type_column = 'Property Type - Self-Selected'
dataframes_to_update = [properties, gas_with_properties, electric_with_properties]

for df in dataframes_to_update:
    df[type_column] = df[type_column].replace(building_type_mapping)

print('Building types were grouped into the new broader categories for all three dataframes.')

Building types were grouped into the new broader categories for all three dataframes.


## Count rows for each new building type

Now that the detailed property types have been grouped into broader categories, this section counts how many rows belong to each new building type in properties, gas_with_properties, and electric_with_properties.

In [12]:
grouped_type_dataframes = {
    'properties': properties,
    'gas_with_properties': gas_with_properties,
    'electric_with_properties': electric_with_properties,
}

for df_name, df in grouped_type_dataframes.items():
    print(f'\n{df_name} grouped building type counts:')
    display(
        df[type_column]
        .value_counts(dropna=False)
        .rename_axis(type_column)
        .reset_index(name='Row Count')
    )


properties grouped building type counts:


,Property Type - Self-Selected,Row Count
0,Other-Public Services,881
1,Transportation,307
2,Recreation & Community,178
3,Public Safety,129
4,Education & Social Services,111
5,Utilities / Infrastructure,71
6,Administrative / Office,54



gas_with_properties grouped building type counts:


,Property Type - Self-Selected,Row Count
0,Other-Public Services,3084
1,Recreation & Community,1860
2,Public Safety,1459
3,Education & Social Services,1221
4,Administrative / Office,516
5,Transportation,286
6,Utilities / Infrastructure,96



electric_with_properties grouped building type counts:


,Property Type - Self-Selected,Row Count
0,Other-Public Services,9867
1,Transportation,3610
2,Recreation & Community,2100
3,Public Safety,1513
4,Education & Social Services,1304
5,Utilities / Infrastructure,840
6,Administrative / Office,598


## Remove selected portfolio manager IDs from gas data

This cell removes rows from `gas_with_properties` where `Portfolio Manager ID` is `35000492` or `34999500`, then returns the updated dataframe shape.

In [14]:
portfolio_manager_ids_to_remove = [35000492, 34999500]

gas_with_properties = gas_with_properties[
    ~gas_with_properties['Portfolio Manager ID'].isin(portfolio_manager_ids_to_remove)
].reset_index(drop=True)

gas_with_properties.shape

(8497, 10)